In [1]:
%pip install -Uq langchain langchain-core langchain-groq


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
from langchain_core.chat_history import InMemoryChatMessageHistory

history = InMemoryChatMessageHistory()

history.add_user_message("안녕하세요? 저는 에디입니다.")
history.add_ai_message("안녕하세요, 에디님! 무엇을 도와드릴까요?")
history.add_user_message("저는 파이썬을 공부하고있어요.")
history.add_ai_message("좋아요! 파이썬은 배우기 좋은 언어입니다. 무엇부터 시작할까요?")

print(len(history.messages))

for msg in history.messages:
    print(msg)

4
content='안녕하세요? 저는 에디입니다.' additional_kwargs={} response_metadata={}
content='안녕하세요, 에디님! 무엇을 도와드릴까요?' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
content='저는 파이썬을 공부하고있어요.' additional_kwargs={} response_metadata={}
content='좋아요! 파이썬은 배우기 좋은 언어입니다. 무엇부터 시작할까요?' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]


In [11]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다. 한국어로 대답해주세요"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

chain = prompt | llm | parser

store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='chat_history'
)


In [12]:
SESSION = "user_001"

MESSAGE_HISTORY = [
    "저는 에디라고 합니다.",
    "제 이름이 뭔지 기억하나요?",
    "저는 요즘 머신러닝을 공부해요",
    "제가 공부하는게 뭐라고 했죠?"
]

for msg in MESSAGE_HISTORY:
    answer = chain_with_history.invoke(
        {"input": msg},
        config={"configurable": {"session_id": SESSION}}
    )

    msg_count = len(store[SESSION].messages)
    print(f" 질문: {msg} ")
    print(f" 답변: {answer}")
    print(f" 누적 메세지 개수: {msg_count}")


 질문: 저는 에디라고 합니다. 
 답변: 안녕하세요! 저는 친절한 AI 어시스턴트입니다. 에디라고 하시군요! 반갑습니다. 무엇에 도움이 필요하신가요?
 누적 메세지 개수: 2
 질문: 제 이름이 뭔지 기억하나요? 
 답변: 네! 에디라고 기억하고 있습니다. 에디님과 함께 채팅을 하면서 도움이 필요하시면 언제든지 말씀해 주세요!
 누적 메세지 개수: 4
 질문: 저는 요즘 머신러닝을 공부해요 
 답변: 머신러닝을 공부하실 때는 많은 도움이 될 것 같은데요. 머신러닝에 대한 궁금한 점이 있는지 어떤 부분을 공부하시고 계신지 알려주세요. 저는 도와드릴 수 있는 부분을 알려드릴게요!
 누적 메세지 개수: 6
 질문: 제가 공부하는게 뭐라고 했죠? 
 답변: 요즘 머신러닝을 공부하고 계신다는 말씀이네요! 그럼, 머신러닝에 관련된 도움을 드릴 수 있습니다. 어떤 부분에 어려움을 느끼시거나, 어떤 개념에 대해 궁금증이 있는지 알려주세요!
 누적 메세지 개수: 8
